# Part 4 Task 2 — UNet segmentation of OASIS brain MRI

A UNet built from scratch, trained to segment OASIS MR brain slices into **four** tissue
classes. The marked requirement is **> 0.9 DSC on every label individually** — not on average —
which is what drives the loss function choice below.

**Staged structure**, each gated by an environment variable so the cheap stages run alone:

| stage | what it does | needs data? | needs GPU? | cost |
|---|---|---|---|---|
| 1 | Architecture sanity check — random noise in, confirm `(B, 4, H, W)` out, gradients flow | no | no | seconds |
| 2 | Smoke test — a few hundred real pairs, 2 epochs, loss drops, show a triptych | yes | no | ~1 min |
| 3 | Full training run, timed, loss curves + per-class validation Dice each epoch | yes | yes | minutes |
| 4 | Test-set evaluation — per-class DSC table, bar chart, qualitative visualisations | yes | yes | ~1 min |
| 5 | Save PNG artifacts + results JSON so a SLURM run leaves evidence behind | yes | yes | seconds |

Stage 1 always runs. Stages 2–5 are controlled by `UNET_STAGE2` … `UNET_STAGE5`.

## Configuration

### The resolution tradeoff

The OASIS slices are 256×256. `UNET_IMG` defaults to **128**, which trains roughly 4× faster and
lets a comfortable batch fit in GPU memory.

This is not free, and it costs exactly the thing being measured. DSC is an **overlap** metric, and
for structures this size most of the disagreement between prediction and ground truth lives in a
one- or two-pixel band along the boundary. Halving the resolution halves the number of boundary
pixels but also doubles what each one is worth, so boundary error translates almost directly into
lost DSC — and it hits the smallest class hardest, because its perimeter-to-area ratio is worst.

So: train at 128 to iterate quickly, and if class 1 lands just short of 0.9, the first thing to
try is `UNET_IMG=256`. `UNET_IMG` must be divisible by 16 (four stride-2 downsamples).

### Class imbalance

| class | share of pixels |
|---|---|
| 0 (background) | 72.25% |
| 1 | **5.76%** ← the hard one |
| 2 | 11.35% |
| 3 | 10.64% |

A model that predicted *only* background would score 72% pixel accuracy and a cross-entropy loss
that looks like it is converging. It would also score DSC 0.0 on every real class. This is why
the loss combines Dice with cross-entropy — see the loss cell.

In [ ]:
import os
import re
import time
import json
import glob
import platform

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image

import matplotlib
matplotlib.use("Agg")      # non-interactive backend: figures save cleanly under nbconvert
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


def env_int(name, default):
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else default


def env_float(name, default):
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else default


def env_flag(name, default=False):
    raw = os.environ.get(name, "").strip().lower()
    return raw in ("1", "true", "yes", "y", "on") if raw else default


# ---------------------------------------------------------------- device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

SEED = env_int("UNET_SEED", 42)
torch.manual_seed(SEED)
np.random.seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------------- data location
OASIS_ROOT = os.environ.get("UNET_DATA", "/home/groups/comp3710/OASIS")

# Image directory paired with its matching segmentation directory.
SPLITS = {
    "train":    (os.path.join(OASIS_ROOT, "keras_png_slices_train"),
                 os.path.join(OASIS_ROOT, "keras_png_slices_seg_train")),
    "validate": (os.path.join(OASIS_ROOT, "keras_png_slices_validate"),
                 os.path.join(OASIS_ROOT, "keras_png_slices_seg_validate")),
    "test":     (os.path.join(OASIS_ROOT, "keras_png_slices_test"),
                 os.path.join(OASIS_ROOT, "keras_png_slices_seg_test")),
}

DATA_AVAILABLE = all(os.path.isdir(d) for pair in SPLITS.values() for d in pair)

# ---------------------------------------------------------------- hyperparameters
NUM_CLASSES = 4
MASK_STEP = 85          # masks store classes as 0, 85, 170, 255 -> divide by 85 for indices

IMG_SIZE = env_int("UNET_IMG", 128)       # must be divisible by 16
BASE_CH = env_int("UNET_BASE", 32)        # first encoder width; 64 is the paper's, 32 is lighter
BATCH_SIZE = env_int("UNET_BATCH", 16)
EPOCHS = env_int("UNET_EPOCHS", 25)
LR = env_float("UNET_LR", 1e-3)

# Loss mixing weights. Both terms are kept — see the loss cell for why neither alone suffices.
CE_WEIGHT = env_float("UNET_CE_W", 0.5)
DICE_WEIGHT = env_float("UNET_DICE_W", 0.5)

NUM_WORKERS = env_int("UNET_WORKERS", 0 if platform.system() == "Windows" else 4)

# ---------------------------------------------------------------- stage gates
RUN_STAGE2 = env_flag("UNET_STAGE2", True)
RUN_STAGE3 = env_flag("UNET_STAGE3", False)
RUN_STAGE4 = env_flag("UNET_STAGE4", False)
RUN_STAGE5 = env_flag("UNET_STAGE5", False)

OUT_DIR = os.environ.get("UNET_OUT", "./unet_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

DSC_TARGET = 0.9        # the lab sheet requirement, applied per label

assert IMG_SIZE % 16 == 0, f"UNET_IMG must be divisible by 16 (four downsamples), got {IMG_SIZE}"

# Discrete colormap for the four labels — a continuous map would imply the class indices
# are ordered on a scale, which they are not.
CLASS_COLOURS = ListedColormap(["#000000", "#e45756", "#54a24b", "#4c78a8"])
CLASS_NAMES = ["0 background", "1 CSF", "2 grey matter", "3 white matter"]

print(f"device        : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
print(f"torch         : {torch.__version__}")
print(f"data root     : {OASIS_ROOT}  (available: {DATA_AVAILABLE})")
print(f"image size    : {IMG_SIZE}x{IMG_SIZE}")
print(f"base channels : {BASE_CH}")
print(f"classes       : {NUM_CLASSES}")
print(f"batch / epochs: {BATCH_SIZE} / {EPOCHS}")
print(f"loss weights  : CE {CE_WEIGHT} + Dice {DICE_WEIGHT}")
print(f"stages 2/3/4/5: {RUN_STAGE2}/{RUN_STAGE3}/{RUN_STAGE4}/{RUN_STAGE5}")
print(f"output dir    : {OUT_DIR}")

## The dataset

### Pairing images to masks

Filenames differ only in their prefix: `case_001_slice_0.nii.png` in the image directory pairs
with `seg_001_slice_0.nii.png` in the segmentation directory. We build the mask path by
substituting the prefix and **assert the file exists**, rather than sorting both directories and
zipping them — sorted-zip pairing fails silently if either directory has a stray or missing file,
and a silently mispaired dataset trains to a plausible-looking loss while learning nothing.

### Decoding the masks

The masks are greyscale PNGs storing four classes as pixel values `0, 85, 170, 255`. Dividing by
85 recovers indices `0..3`. We round rather than floor-divide so a value off by one (from any
resampling upstream) still lands on the right class instead of silently dropping a level.

### Why NEAREST for masks and BILINEAR for images

This is the single most important detail in this cell. Bilinear interpolation **averages**
neighbouring pixels. On an image that is correct — intensities are continuous. On a label map it
is meaningless: averaging a class-1 pixel (85) with a class-2 pixel (170) gives 127, which
decodes to class 1.5 and then rounds into a class that was never there. NEAREST picks a single
original pixel, so only real label values survive.

### No augmentation

OASIS slices are spatially registered, and flipping would break the left/right anatomical
correspondence the model can otherwise rely on. Same reasoning as Task 1.

In [ ]:
class OASISSegDataset(Dataset):
    """Paired OASIS MR slices and their 4-class segmentation masks.

    Returns (image, mask) where:
        image : float32, shape (1, S, S), scaled to [0, 1]
        mask  : int64,   shape (S, S),    values in {0, 1, 2, 3}

    The mask is returned as integer class indices, not one-hot. `nn.CrossEntropyLoss`
    wants indices, and the Dice term one-hots internally — carrying indices around keeps
    the tensors 4x smaller.
    """

    def __init__(self, image_dir, mask_dir, img_size=128, limit=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size

        # sorted() so ordering is deterministic across machines; raw glob order is
        # filesystem-dependent and would quietly break reproducibility.
        self.image_paths = sorted(glob.glob(os.path.join(image_dir, "*.png")))
        if limit is not None:
            self.image_paths = self.image_paths[:limit]
        if not self.image_paths:
            raise FileNotFoundError(f"no PNGs found in {image_dir}")

        # Build each mask path by prefix substitution, and verify it exists.
        self.mask_paths = []
        missing = []
        for p in self.image_paths:
            name = os.path.basename(p)
            # "case_001_slice_0.nii.png" -> "seg_001_slice_0.nii.png"
            mask_name = re.sub(r"^case_", "seg_", name)
            mask_path = os.path.join(mask_dir, mask_name)
            if not os.path.isfile(mask_path):
                missing.append(mask_name)
            self.mask_paths.append(mask_path)

        assert not missing, (
            f"{len(missing)} images have no matching mask in {mask_dir}; "
            f"first few: {missing[:5]}")
        assert len(self.image_paths) == len(self.mask_paths), (
            f"count mismatch: {len(self.image_paths)} images vs {len(self.mask_paths)} masks")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, i):
        # ---- image: continuous intensities, BILINEAR is correct ----
        img = Image.open(self.image_paths[i]).convert("L")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).unsqueeze(0)

        # ---- mask: discrete labels, NEAREST is mandatory ----
        m = Image.open(self.mask_paths[i]).convert("L")
        if m.size != (self.img_size, self.img_size):
            m = m.resize((self.img_size, self.img_size), Image.NEAREST)

        arr = np.asarray(m, dtype=np.float32)
        # 0, 85, 170, 255 -> 0, 1, 2, 3. Round (not floor) so an off-by-one value still
        # lands on the intended class; clip guards against anything unexpected.
        arr = np.clip(np.round(arr / MASK_STEP), 0, NUM_CLASSES - 1)
        y = torch.from_numpy(arr).long()

        return x, y


def make_loader(split, batch_size, shuffle, limit=None):
    """Build a Dataset + DataLoader for one named split."""
    image_dir, mask_dir = SPLITS[split]
    ds = OASISSegDataset(image_dir, mask_dir, img_size=IMG_SIZE, limit=limit)
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=shuffle,          # only drop the ragged last batch when training
    )
    return ds, loader


if DATA_AVAILABLE:
    _probe = OASISSegDataset(*SPLITS["train"], img_size=IMG_SIZE, limit=64)
    _x, _y = _probe[0]
    print(f"train pairs found: {len(glob.glob(os.path.join(SPLITS['train'][0], '*.png'))):,}")
    print(f"image tensor     : {tuple(_x.shape)}  dtype {_x.dtype}  "
          f"range [{_x.min():.3f}, {_x.max():.3f}]")
    print(f"mask tensor      : {tuple(_y.shape)}  dtype {_y.dtype}  "
          f"labels {sorted(torch.unique(_y).tolist())}")

    # Measure the actual class balance on the probe subset — confirms the mask decoding is
    # right and reproduces the distribution quoted in the lab sheet.
    counts = torch.zeros(NUM_CLASSES)
    for k in range(len(_probe)):
        _, yy = _probe[k]
        counts += torch.bincount(yy.flatten(), minlength=NUM_CLASSES).float()
    share = counts / counts.sum()
    print("\nclass balance on probe subset:")
    for c in range(NUM_CLASSES):
        print(f"  {CLASS_NAMES[c]:<18} {share[c] * 100:5.2f}%")
else:
    print(f"Data not found under {OASIS_ROOT}")
    print("Stage 1 will still run (it needs no data). Stages 2-5 will skip.")

## The UNet

### Why skip connections exist

A plain encoder–decoder downsamples an image to a small, deep bottleneck and then upsamples back.
The encoder answers *what* is in the image; the problem is that downsampling throws away *where*.
By the bottleneck a 128×128 input is 8×8 — each cell covers a 16×16 patch of the original, so all
positional information finer than 16 pixels is simply gone. The decoder cannot invent it back,
and the output comes out as blobs in roughly the right place with mushy, imprecise boundaries.

That is fatal here, because DSC is dominated by boundary accuracy.

A **skip connection** carries the encoder's feature map at each resolution straight across to the
matching decoder level, where it is **concatenated** onto the upsampled features:

```
   enc1 (128x128, 32ch) ───────────────────────────────► concat ─► dec1 ─► out
     │ pool                                                ▲
   enc2 (64x64, 64ch) ─────────────────────► concat ─► dec2
     │ pool                                    ▲
   enc3 (32x32, 128ch) ──────► concat ─► dec3 ─┘
     │ pool                      ▲
   enc4 (16x16, 256ch) ► concat ─┘
     │ pool
   bottleneck (8x8, 512ch)
```

So each decoder level gets both the *semantic* signal coming up from below ("this region is grey
matter") and the *high-resolution* signal coming across ("the edge is exactly here"). That is what
makes precise boundaries possible, and it is the whole point of the architecture.

Note it is **concatenation**, not addition as in a ResNet. ResNet adds because it wants to learn a
residual correction to the same representation. UNet concatenates because the two sources are
different *kinds* of information, and the following conv should be free to weigh them separately.

### Categorical (one-hot) output

The final layer is a 1×1 conv producing **`NUM_CLASSES` = 4 channels**, one logit per class per
pixel — the categorical formulation the lab sheet requires. Softmax over the channel axis gives a
per-pixel probability distribution across the four classes; `argmax` gives the predicted label.
The 1×1 kernel is right because this is a per-pixel decision: all spatial reasoning already
happened, and this step just projects each pixel's feature vector onto the class axes.

No softmax is applied inside the model — `CrossEntropyLoss` and the Dice term both expect raw
logits and apply their own.

In [ ]:
class DoubleConv(nn.Module):
    """(conv 3x3 -> BN -> ReLU) x 2, the repeating unit of both UNet paths.

    Two stacked 3x3 convolutions see a 5x5 receptive field using fewer parameters than one
    5x5 would, with an extra nonlinearity in between. BatchNorm keeps activations scaled and
    lets us train at a higher learning rate; bias=False because BN's shift subsumes it.
    """

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),   # padding=1 keeps H,W
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """UNet for single-channel input and NUM_CLASSES-way per-pixel classification.

    forward: (B, 1, S, S) -> (B, num_classes, S, S) raw logits
    """

    def __init__(self, in_ch=1, num_classes=4, base_ch=32):
        super().__init__()

        c1, c2, c3, c4 = base_ch, base_ch * 2, base_ch * 4, base_ch * 8
        c5 = base_ch * 16                       # bottleneck

        # ---------------- encoder (contracting path) ----------------
        # Each level: DoubleConv, keep the output for the skip, then pool to halve resolution.
        self.enc1 = DoubleConv(in_ch, c1)       # S
        self.enc2 = DoubleConv(c1, c2)          # S/2
        self.enc3 = DoubleConv(c2, c3)          # S/4
        self.enc4 = DoubleConv(c3, c4)          # S/8
        self.pool = nn.MaxPool2d(2)             # stateless, so one instance is reused

        # ---------------- bottleneck ----------------
        self.bottleneck = DoubleConv(c4, c5)    # S/16

        # ---------------- decoder (expanding path) ----------------
        # ConvTranspose2d with kernel 2 stride 2 exactly doubles H and W. A learned upsample
        # rather than a fixed interpolation, so the network can decide how to fill in detail.
        self.up4 = nn.ConvTranspose2d(c5, c4, 2, stride=2)
        # in_ch is c4*2 because we concatenate the skip (c4) onto the upsampled map (c4).
        self.dec4 = DoubleConv(c4 * 2, c4)

        self.up3 = nn.ConvTranspose2d(c4, c3, 2, stride=2)
        self.dec3 = DoubleConv(c3 * 2, c3)

        self.up2 = nn.ConvTranspose2d(c3, c2, 2, stride=2)
        self.dec2 = DoubleConv(c2 * 2, c2)

        self.up1 = nn.ConvTranspose2d(c2, c1, 2, stride=2)
        self.dec1 = DoubleConv(c1 * 2, c1)

        # ---------------- output head ----------------
        # 1x1 conv: a per-pixel linear projection onto the class axes. One channel per class
        # = the categorical (one-hot) output the lab sheet asks for.
        self.out_conv = nn.Conv2d(c1, num_classes, kernel_size=1)

    def forward(self, x):
        # ---- encoder, keeping each level's output for its skip connection ----
        s1 = self.enc1(x)                    # (B, c1,  S,    S)
        s2 = self.enc2(self.pool(s1))        # (B, c2,  S/2,  S/2)
        s3 = self.enc3(self.pool(s2))        # (B, c3,  S/4,  S/4)
        s4 = self.enc4(self.pool(s3))        # (B, c4,  S/8,  S/8)

        b = self.bottleneck(self.pool(s4))   # (B, c5,  S/16, S/16)

        # ---- decoder: upsample, concatenate the matching skip, then convolve ----
        # dim=1 is the channel axis, so concat stacks feature maps rather than batching them.
        d4 = self.dec4(torch.cat([self.up4(b), s4], dim=1))    # (B, c4, S/8,  S/8)
        d3 = self.dec3(torch.cat([self.up3(d4), s3], dim=1))   # (B, c3, S/4,  S/4)
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))   # (B, c2, S/2,  S/2)
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))   # (B, c1, S,    S)

        return self.out_conv(d1)             # (B, num_classes, S, S) — raw logits

## The loss — why Dice, and why not cross-entropy alone

### What cross-entropy optimises

CE averages a per-pixel penalty over every pixel in the batch. Because class 0 is **72%** of
pixels and class 1 is **5.76%**, background contributes about 12× more to the gradient. The
cheapest way for the model to reduce CE early is to get background right everywhere and treat the
small classes as noise. CE will keep falling while class 1 DSC sits near zero — the loss curve
looks healthy and the model is failing the actual requirement.

Class weighting (`CrossEntropyLoss(weight=...)`) partly fixes this, but it needs hand-tuning and
still optimises a per-pixel proxy rather than the overlap we are graded on.

### What Dice optimises

The Dice coefficient for one class is

```
  DSC = 2 |P ∩ G| / (|P| + |G|)
```

— a **ratio**, normalised by the size of that class. A class occupying 5% of the image and one
occupying 72% both produce a score in [0,1], and we average those scores over classes. Every class
therefore contributes **equally** to the loss regardless of how many pixels it owns. That is the
property CE lacks, and it is exactly what a per-label DSC requirement needs.

Dice is also directly the metric being graded, so we are optimising the thing we are measured on
rather than a proxy for it.

### Why keep both

Dice alone is unstable early in training: when predictions are near-random the numerator is tiny,
the gradient is noisy, and a class absent from a batch gives a degenerate 0/0. CE provides a
smooth, well-conditioned per-pixel gradient that gets the model into a sensible region quickly.
So we use both:

```
  loss = CE_WEIGHT * CE  +  DICE_WEIGHT * (1 - mean per-class Dice)
```

CE supplies stable early gradients and per-pixel confidence; Dice supplies the class-balanced
overlap objective. This combination is standard practice in medical image segmentation.

### The smoothing term

`eps` in the numerator and denominator prevents a 0/0 when a class is absent from the batch
(common for class 1 on slices near the top or bottom of the brain) — with it, correctly predicting
"absent" scores 1 rather than NaN.

In [ ]:
def soft_dice_loss(logits, target, eps=1.0):
    """Differentiable Dice loss over all classes.

    Uses softmax probabilities rather than a hard argmax, because argmax has zero gradient
    almost everywhere and would make the term untrainable.

    logits: (B, C, H, W) raw scores
    target: (B, H, W)    integer class indices
    """
    probs = F.softmax(logits, dim=1)                              # (B, C, H, W)

    # One-hot the integer targets so they line up with the per-class probability channels.
    # one_hot gives (B, H, W, C); permute moves C to axis 1 to match `probs`.
    target_1h = F.one_hot(target, probs.shape[1]).permute(0, 3, 1, 2).float()

    # Sum over batch, height and width — leaving one number per CLASS. Aggregating the batch
    # into a single Dice per class (rather than per image, then averaged) is more stable when
    # a class is missing from some images.
    dims = (0, 2, 3)
    intersection = (probs * target_1h).sum(dims)                  # (C,)
    cardinality = probs.sum(dims) + target_1h.sum(dims)           # (C,)

    dice_per_class = (2.0 * intersection + eps) / (cardinality + eps)

    # Mean over classes: every class weighs the same, whatever its pixel count.
    return 1.0 - dice_per_class.mean()


class CombinedLoss(nn.Module):
    """CE_WEIGHT * CrossEntropy + DICE_WEIGHT * DiceLoss.

    Returns (total, ce, dice) so the two components can be logged separately — useful for
    diagnosing which term has stalled.
    """

    def __init__(self, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
        # CrossEntropyLoss takes raw logits and INTEGER targets; it applies log_softmax itself.
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, target):
        ce = self.ce(logits, target)
        dice = soft_dice_loss(logits, target)
        return self.ce_weight * ce + self.dice_weight * dice, ce, dice


@torch.no_grad()
def update_dice_counts(logits, target, intersection, cardinality, num_classes=NUM_CLASSES):
    """Accumulate hard-Dice counts over a dataset, in place.

    This is the EVALUATION metric, separate from the loss: it uses a hard argmax, which is
    what actually gets reported. Accumulating intersection and cardinality across the whole
    split and dividing once at the end gives the aggregate DSC — more stable than averaging
    per-image scores, where a slice containing none of a class yields a degenerate value.
    """
    pred = logits.argmax(dim=1)                    # (B, H, W)
    for c in range(num_classes):
        p = pred == c
        g = target == c
        intersection[c] += (p & g).sum().item()
        cardinality[c] += p.sum().item() + g.sum().item()


def dice_from_counts(intersection, cardinality):
    """Turn accumulated counts into per-class DSC. Absent-and-not-predicted scores 1.0."""
    dsc = np.ones(len(intersection), dtype=np.float64)
    for c in range(len(intersection)):
        if cardinality[c] > 0:
            dsc[c] = 2.0 * intersection[c] / cardinality[c]
    return dsc

## Stage 1 — architecture sanity check

No data, no GPU. Confirms the output is `(B, 4, H, W)` — same spatial size as the input, one
channel per class — that the parameter count is sane, and that gradients reach every layer.

A UNet has a specific failure mode worth checking explicitly: if a skip connection is wired to
the wrong level, `torch.cat` raises a size mismatch. That is what makes this two-second check
worth running before any GPU time.

In [ ]:
print("=" * 66)
print("STAGE 1 — architecture sanity check (random noise, no data)")
print("=" * 66)

model_check = UNet(in_ch=1, num_classes=NUM_CLASSES, base_ch=BASE_CH)
model_check.eval()

# torch.rand (uniform [0,1]) rather than torch.randn: real inputs are scaled to [0,1], so this
# matches the actual input distribution the model will see.
dummy = torch.rand(2, 1, IMG_SIZE, IMG_SIZE)

with torch.no_grad():
    logits = model_check(dummy)

print(f"input  shape : {tuple(dummy.shape)}")
print(f"output shape : {tuple(logits.shape)}")

# The defining contract of a segmentation net: one logit per class per input pixel.
assert logits.shape == (2, NUM_CLASSES, IMG_SIZE, IMG_SIZE), (
    f"expected (2, {NUM_CLASSES}, {IMG_SIZE}, {IMG_SIZE}), got {tuple(logits.shape)}")

# Softmax over the channel axis must give a valid distribution per pixel.
probs = F.softmax(logits, dim=1)
assert torch.allclose(probs.sum(dim=1), torch.ones(2, IMG_SIZE, IMG_SIZE), atol=1e-5), \
    "per-pixel class probabilities do not sum to 1"
print(f"per-pixel probabilities sum to 1: OK")
print(f"argmax label range: {probs.argmax(1).min().item()}..{probs.argmax(1).max().item()} "
      f"(valid: 0..{NUM_CLASSES - 1})")

# --- trace resolutions through the encoder, confirming the skips line up ---
print("\nencoder feature maps (each is a skip connection source):")
with torch.no_grad():
    s1 = model_check.enc1(dummy)
    s2 = model_check.enc2(model_check.pool(s1))
    s3 = model_check.enc3(model_check.pool(s2))
    s4 = model_check.enc4(model_check.pool(s3))
    b = model_check.bottleneck(model_check.pool(s4))
for name, t in (("enc1", s1), ("enc2", s2), ("enc3", s3), ("enc4", s4), ("bottleneck", b)):
    print(f"  {name:<11}: {tuple(t.shape)}")

n_params = sum(p.numel() for p in model_check.parameters() if p.requires_grad)
print(f"\ntrainable parameters: {n_params:,}")

# --- gradients reach every layer ---
model_check.train()
criterion_check = CombinedLoss(CE_WEIGHT, DICE_WEIGHT)
fake_target = torch.randint(0, NUM_CLASSES, (2, IMG_SIZE, IMG_SIZE))
total, ce, dice = criterion_check(model_check(dummy), fake_target)
total.backward()
missing = [n for n, p in model_check.named_parameters() if p.requires_grad and p.grad is None]
assert not missing, f"parameters with no gradient: {missing[:5]}"
print(f"loss on random input: total {total.item():.4f}  ce {ce.item():.4f}  dice {dice.item():.4f}")

print("\nSTAGE 1 PASSED — output shape, probability normalisation and gradient flow all correct.")
del model_check

## Training and evaluation functions

Shared by the smoke test and the full run, so stage 2 exercises exactly the code stage 3 uses.
`evaluate` returns per-class DSC rather than a single average, because that is the requirement.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler=None):
    """One pass over `loader`. Returns (total, ce, dice) means and per-class train DSC."""
    model.train()
    tot = ce_tot = dice_tot = 0.0
    n = 0
    inter = np.zeros(NUM_CLASSES)
    card = np.zeros(NUM_CLASSES)

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss, ce, dice = criterion(logits, y)

        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()          # OneCycleLR steps per BATCH, not per epoch

        b = x.size(0)
        tot += loss.item() * b
        ce_tot += ce.item() * b
        dice_tot += dice.item() * b
        n += b
        update_dice_counts(logits, y, inter, card)

    return tot / n, ce_tot / n, dice_tot / n, dice_from_counts(inter, card)


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate on a held-out split. Returns (total, ce, dice) means and per-class DSC."""
    model.eval()
    tot = ce_tot = dice_tot = 0.0
    n = 0
    inter = np.zeros(NUM_CLASSES)
    card = np.zeros(NUM_CLASSES)

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)
        loss, ce, dice = criterion(logits, y)

        b = x.size(0)
        tot += loss.item() * b
        ce_tot += ce.item() * b
        dice_tot += dice.item() * b
        n += b
        update_dice_counts(logits, y, inter, card)

    return tot / n, ce_tot / n, dice_tot / n, dice_from_counts(inter, card)


def show_triptychs(model, loader, n_show, path, title):
    """Save rows of (MR image | ground truth | prediction) for qualitative inspection."""
    model.eval()
    x, y = next(iter(loader))
    n_show = min(n_show, x.size(0))

    with torch.no_grad():
        pred = model(x[:n_show].to(DEVICE)).argmax(dim=1).cpu()

    fig, axes = plt.subplots(n_show, 3, figsize=(9, 3 * n_show))
    axes = np.atleast_2d(axes)
    for i in range(n_show):
        axes[i, 0].imshow(x[i, 0], cmap="gray", vmin=0, vmax=1)
        axes[i, 1].imshow(y[i], cmap=CLASS_COLOURS, vmin=0, vmax=NUM_CLASSES - 1, interpolation="nearest")
        axes[i, 2].imshow(pred[i], cmap=CLASS_COLOURS, vmin=0, vmax=NUM_CLASSES - 1, interpolation="nearest")
        for j, lbl in enumerate(("MR image", "ground truth", "prediction")):
            axes[i, j].axis("off")
            if i == 0:
                axes[i, j].set_title(lbl)

    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    print(f"saved -> {path}")
    plt.show()
    plt.close(fig)


def print_dsc_table(dsc, label="DSC"):
    """Print per-class DSC against the 0.9 requirement. Returns True if all labels pass."""
    print(f"\n{'class':<20}{label:>9}{'':>4}{'status':>8}")
    print("-" * 42)
    for c in range(NUM_CLASSES):
        ok = dsc[c] >= DSC_TARGET
        print(f"{CLASS_NAMES[c]:<20}{dsc[c]:>9.4f}{'':>4}{'PASS' if ok else 'FAIL':>8}")
    print("-" * 42)
    print(f"{'mean':<20}{dsc.mean():>9.4f}")
    all_pass = bool((dsc >= DSC_TARGET).all())
    print(f"\nAll labels > {DSC_TARGET}: {'YES' if all_pass else 'NO'}")
    return all_pass

## Stage 2 — smoke test

Real image/mask pairs, but only a few hundred of them for 2 epochs. Not trying to learn anything
useful — it answers three cheap questions before committing to a GPU queue:

1. Does the pairing work, and do the tensors have the right shapes and dtypes?
2. Does the loss go **down**, i.e. is the optimiser wired up correctly?
3. Does anything produce NaN or inf?

The subset is deliberately **512 images** rather than a handful: with `batch_size=8` that gives
64 optimiser steps per epoch, which is enough for a per-batch learning-rate scheduler to be
meaningful. A subset of one or two batches would make `OneCycleLR` degenerate.

After 2 epochs the prediction will be crude — mostly background with rough blobs. That is
expected. What matters is that it is *spatially organised* rather than noise.

In [ ]:
print("=" * 66)
print("STAGE 2 — smoke test")
print("=" * 66)

smoke_model = None
if not RUN_STAGE2:
    print("SKIPPED — UNET_STAGE2 is off.")
elif not DATA_AVAILABLE:
    print(f"SKIPPED — no data under {OASIS_ROOT}.")
else:
    SMOKE_IMAGES = 512
    SMOKE_EPOCHS = 2
    SMOKE_BATCH = 8

    smoke_ds, smoke_loader = make_loader("train", SMOKE_BATCH, shuffle=True, limit=SMOKE_IMAGES)
    print(f"smoke subset: {len(smoke_ds)} pairs, {len(smoke_loader)} batches/epoch, "
          f"{len(smoke_loader) * SMOKE_EPOCHS} optimiser steps total")
    assert len(smoke_loader) >= 8, "smoke subset too small for a per-batch LR scheduler"

    xb, yb = next(iter(smoke_loader))
    print(f"image batch : {tuple(xb.shape)}  dtype {xb.dtype}  range [{xb.min():.3f}, {xb.max():.3f}]")
    print(f"mask batch  : {tuple(yb.shape)}  dtype {yb.dtype}  labels {sorted(torch.unique(yb).tolist())}")
    assert xb.shape[1:] == (1, IMG_SIZE, IMG_SIZE), f"unexpected image shape {tuple(xb.shape)}"
    assert yb.shape[1:] == (IMG_SIZE, IMG_SIZE), f"unexpected mask shape {tuple(yb.shape)}"
    assert yb.max() < NUM_CLASSES, f"mask contains label {yb.max().item()} >= {NUM_CLASSES}"

    smoke_model = UNet(1, NUM_CLASSES, BASE_CH).to(DEVICE)
    smoke_criterion = CombinedLoss(CE_WEIGHT, DICE_WEIGHT)
    smoke_opt = torch.optim.Adam(smoke_model.parameters(), lr=LR)

    smoke_losses = []
    for epoch in range(1, SMOKE_EPOCHS + 1):
        t, ce, d, dsc = train_one_epoch(smoke_model, smoke_loader, smoke_criterion, smoke_opt)
        smoke_losses.append(t)
        print(f"  epoch {epoch}: total {t:.4f}  ce {ce:.4f}  dice {d:.4f}  "
              f"train DSC {np.round(dsc, 3).tolist()}")

    assert all(np.isfinite(smoke_losses)), f"loss went non-finite: {smoke_losses}"
    assert smoke_losses[-1] < smoke_losses[0], (
        f"loss did not decrease ({smoke_losses[0]:.4f} -> {smoke_losses[-1]:.4f}); "
        "check the optimiser and that zero_grad is being called")

    show_triptychs(smoke_model, smoke_loader, n_show=2,
                   path=os.path.join(OUT_DIR, "stage2_smoke_triptych.png"),
                   title="Stage 2: after 2 epochs on 512 images (expected to be crude)")

    print(f"\nSTAGE 2 PASSED — loss {smoke_losses[0]:.4f} -> {smoke_losses[-1]:.4f}, all finite.")

## Stage 3 — full training run

All 9,664 training pairs for `EPOCHS` epochs, with per-class **validation** DSC computed every
epoch. Watching all four classes separately is the point: the mean can sit comfortably above 0.9
while class 1 is still failing, and the mean is not what is graded.

Adam with `OneCycleLR`, which steps per batch — warm up, then anneal to near zero. The high
mid-training learning rate acts as a regulariser and the low final rate lets the boundaries
settle, which is where the last few DSC points come from.

We keep the checkpoint with the best **minimum-across-classes** validation DSC, not the best
mean — again because the requirement is per-label, so the useful model is the one whose *worst*
class is best.

Gated behind `UNET_STAGE3=1`.

In [ ]:
print("=" * 66)
print("STAGE 3 — full training run")
print("=" * 66)

model = None
history = None

if not RUN_STAGE3:
    print("SKIPPED — set UNET_STAGE3=1 to run this stage.")
elif not DATA_AVAILABLE:
    print(f"SKIPPED — no data under {OASIS_ROOT}.")
else:
    train_ds, train_loader = make_loader("train", BATCH_SIZE, shuffle=True)
    val_ds, val_loader = make_loader("validate", BATCH_SIZE, shuffle=False)
    print(f"train: {len(train_ds):,} pairs / {len(train_loader)} batches")
    print(f"val  : {len(val_ds):,} pairs / {len(val_loader)} batches")

    model = UNet(1, NUM_CLASSES, BASE_CH).to(DEVICE)
    criterion = CombinedLoss(CE_WEIGHT, DICE_WEIGHT)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=EPOCHS, pct_start=0.25)

    history = {"train_total": [], "train_ce": [], "train_dice": [],
               "val_total": [], "val_ce": [], "val_dice": [],
               "val_dsc": [], "epoch_time": []}

    best_min_dsc = -1.0
    ckpt_path = os.path.join(OUT_DIR, "unet_oasis.pt")

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()      # CUDA is async; sync or we time launches, not work
    t_start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()

        tr_t, tr_ce, tr_d, _ = train_one_epoch(model, train_loader, criterion, optimizer, scheduler)
        va_t, va_ce, va_d, va_dsc = evaluate(model, val_loader, criterion)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        dt = time.perf_counter() - t0

        history["train_total"].append(tr_t)
        history["train_ce"].append(tr_ce)
        history["train_dice"].append(tr_d)
        history["val_total"].append(va_t)
        history["val_ce"].append(va_ce)
        history["val_dice"].append(va_d)
        history["val_dsc"].append(va_dsc.tolist())
        history["epoch_time"].append(dt)

        # flush=True so the line reaches the SLURM log immediately, not at job end.
        print(f"epoch {epoch:3d}/{EPOCHS}  train {tr_t:.4f}  val {va_t:.4f}  "
              f"val DSC [{', '.join(f'{d:.3f}' for d in va_dsc)}]  "
              f"min {va_dsc.min():.3f}  {dt:.1f}s", flush=True)

        # Select on the WORST class, since every label must clear 0.9.
        if va_dsc.min() > best_min_dsc:
            best_min_dsc = va_dsc.min()
            torch.save({"state_dict": model.state_dict(), "img_size": IMG_SIZE,
                        "base_ch": BASE_CH, "num_classes": NUM_CLASSES,
                        "epoch": epoch, "val_dsc": va_dsc.tolist()}, ckpt_path)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    total_time = time.perf_counter() - t_start
    print(f"\ntrained in {total_time:.1f}s ({np.mean(history['epoch_time']):.1f}s/epoch)")
    print(f"best min-class val DSC: {best_min_dsc:.4f}")
    print(f"checkpoint -> {ckpt_path} ({os.path.getsize(ckpt_path) / 1e6:.1f} MB)")

    # Reload the best checkpoint so stages 4-5 evaluate the selected model, not the last epoch.
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)["state_dict"])

    with open(os.path.join(OUT_DIR, "unet_history.json"), "w") as f:
        json.dump(history, f, indent=2)

    # --- curves ---
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    ep = range(1, EPOCHS + 1)

    axes[0].plot(ep, history["train_total"], label="train")
    axes[0].plot(ep, history["val_total"], "--", label="val")
    axes[0].set_title("Combined loss")

    axes[1].plot(ep, history["train_ce"], label="train CE")
    axes[1].plot(ep, history["val_ce"], "--", label="val CE")
    axes[1].plot(ep, history["train_dice"], label="train Dice loss")
    axes[1].plot(ep, history["val_dice"], "--", label="val Dice loss")
    axes[1].set_title("Loss components")

    val_dsc_arr = np.array(history["val_dsc"])       # (epochs, num_classes)
    for c in range(NUM_CLASSES):
        axes[2].plot(ep, val_dsc_arr[:, c], label=CLASS_NAMES[c])
    axes[2].axhline(DSC_TARGET, color="red", ls=":", label=f"{DSC_TARGET} requirement")
    axes[2].set_ylim(0, 1)
    axes[2].set_title("Validation DSC per class")

    for ax in axes:
        ax.set_xlabel("epoch"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "stage3_curves.png"), dpi=120)
    print(f"saved -> {os.path.join(OUT_DIR, 'stage3_curves.png')}")
    plt.show()
    plt.close(fig)

## Stage 4 — test-set evaluation

**This is the graded deliverable.** Per-class DSC on the held-out test set, reported for all four
labels separately against the 0.9 requirement, plus a bar chart and side-by-side visualisations
to justify the numbers qualitatively.

This is also the cell to run live at the demo: it loads the checkpoint, runs inference on the
test split, and prints the table.

In [ ]:
print("=" * 66)
print("STAGE 4 — test-set evaluation")
print("=" * 66)

test_dsc = None
eval_model = model if model is not None else smoke_model

if not RUN_STAGE4:
    print("SKIPPED — set UNET_STAGE4=1 to run this stage.")
elif not DATA_AVAILABLE:
    print(f"SKIPPED — no data under {OASIS_ROOT}.")
else:
    # If no model is in memory (e.g. a fresh kernel at demo time), load the checkpoint.
    if eval_model is None:
        ckpt_path = os.path.join(OUT_DIR, "unet_oasis.pt")
        if os.path.isfile(ckpt_path):
            ck = torch.load(ckpt_path, map_location=DEVICE)
            eval_model = UNet(1, ck["num_classes"], ck["base_ch"]).to(DEVICE)
            eval_model.load_state_dict(ck["state_dict"])
            print(f"loaded checkpoint from epoch {ck['epoch']} -> {ckpt_path}")
        else:
            print("SKIPPED — no model in memory and no checkpoint on disk.")

    if eval_model is not None:
        test_ds, test_loader = make_loader("test", BATCH_SIZE, shuffle=False)
        print(f"test: {len(test_ds):,} pairs / {len(test_loader)} batches")

        criterion_eval = CombinedLoss(CE_WEIGHT, DICE_WEIGHT)
        te_t, te_ce, te_d, test_dsc = evaluate(eval_model, test_loader, criterion_eval)
        print(f"\ntest loss: total {te_t:.4f}  ce {te_ce:.4f}  dice {te_d:.4f}")

        all_pass = print_dsc_table(test_dsc, label="test DSC")

        # --- bar chart ---
        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(range(NUM_CLASSES), test_dsc,
                      color=["#000000", "#e45756", "#54a24b", "#4c78a8"], edgecolor="black")
        ax.axhline(DSC_TARGET, color="red", ls="--", lw=2, label=f"{DSC_TARGET} requirement")
        for i, v in enumerate(test_dsc):
            ax.text(i, v + 0.015, f"{v:.4f}", ha="center", fontweight="bold")
        ax.set_xticks(range(NUM_CLASSES))
        ax.set_xticklabels([n.replace(" ", "\n", 1) for n in CLASS_NAMES])
        ax.set_ylim(0, 1.08)
        ax.set_ylabel("Dice similarity coefficient")
        ax.set_title("Per-class DSC on the OASIS test set")
        ax.legend()
        ax.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        bar_path = os.path.join(OUT_DIR, "stage4_dsc_bars.png")
        plt.savefig(bar_path, dpi=130, bbox_inches="tight")
        print(f"saved -> {bar_path}")
        plt.show()
        plt.close(fig)

        # --- qualitative results ---
        show_triptychs(eval_model, test_loader, n_show=4,
                       path=os.path.join(OUT_DIR, "stage4_test_triptychs.png"),
                       title="Test set: MR image / ground truth / prediction")

## Stage 5 — save artifacts

Writes a results JSON and an extra batch of qualitative examples, so a non-interactive SLURM run
leaves inspectable evidence behind without anyone needing to reopen the notebook.

In [ ]:
print("=" * 66)
print("STAGE 5 — save artifacts")
print("=" * 66)

if not RUN_STAGE5:
    print("SKIPPED — set UNET_STAGE5=1 to run this stage.")
elif test_dsc is None:
    print("SKIPPED — no test results. Run stage 4 first.")
else:
    results = {
        "img_size": IMG_SIZE,
        "base_ch": BASE_CH,
        "num_classes": NUM_CLASSES,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "ce_weight": CE_WEIGHT,
        "dice_weight": DICE_WEIGHT,
        "class_names": CLASS_NAMES,
        "test_dsc_per_class": {CLASS_NAMES[c]: float(test_dsc[c]) for c in range(NUM_CLASSES)},
        "test_dsc_mean": float(test_dsc.mean()),
        "test_dsc_min": float(test_dsc.min()),
        "dsc_target": DSC_TARGET,
        "all_labels_pass": bool((test_dsc >= DSC_TARGET).all()),
    }
    if history is not None:
        results["train_seconds"] = float(np.sum(history["epoch_time"]))
        results["mean_epoch_seconds"] = float(np.mean(history["epoch_time"]))

    res_path = os.path.join(OUT_DIR, "unet_results.json")
    with open(res_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"saved -> {res_path}")
    print(json.dumps(results["test_dsc_per_class"], indent=2))

    # A second, larger set of qualitative examples for the report.
    if eval_model is not None and DATA_AVAILABLE:
        _, extra_loader = make_loader("test", 8, shuffle=False)
        show_triptychs(eval_model, extra_loader, n_show=6,
                       path=os.path.join(OUT_DIR, "stage5_qualitative.png"),
                       title="Additional test-set segmentation results")

    print(f"\nartifacts in {OUT_DIR}/")
    for f in sorted(os.listdir(OUT_DIR)):
        print(f"  {f}")

## Q&A — likely demonstrator questions

**What do the skip connections do?**
The encoder destroys spatial precision as it downsamples — by the bottleneck a 128×128 image is
8×8, so everything finer than a 16-pixel block is gone. The decoder cannot recover it, and a plain
encoder–decoder therefore produces blobs with mushy boundaries. Skip connections carry each
encoder feature map straight across to the decoder level at the same resolution, where it is
concatenated onto the upsampled features. Each decoder level then has both the semantic signal
from below ("this is grey matter") and the high-resolution signal from across ("the edge is
exactly here"). Since DSC is dominated by boundary accuracy, this is where most of the score
comes from.

**Why concatenate rather than add, like ResNet?**
ResNet adds because it is learning a residual correction to the *same* representation. Here the
two inputs are different kinds of information — coarse semantics and fine spatial detail — so we
stack them on the channel axis and let the following convolution learn how to weigh them.

**Why Dice loss instead of plain cross-entropy?**
CE averages a penalty over pixels, and background is 72% of pixels while class 1 is 5.76%. So
background dominates the gradient by roughly 12×, and the cheapest early win is to predict
background everywhere and treat the small classes as noise — CE keeps falling while class 1 DSC
sits near zero. Dice is a *ratio* normalised by each class's own size, so a 5% class and a 72%
class both score in [0,1] and contribute equally once averaged over classes. It is also directly
the metric being graded, so we optimise the objective rather than a proxy.

**Then why keep cross-entropy at all?**
Dice alone is unstable early: with near-random predictions the numerator is tiny, gradients are
noisy, and a class absent from a batch gives a degenerate 0/0. CE provides a smooth,
well-conditioned per-pixel gradient that gets the model into a sensible region fast. The
combination is standard practice in medical segmentation.

**What does DSC measure, and how is it different from pixel accuracy?**
`DSC = 2|P ∩ G| / (|P| + |G|)` — the overlap between predicted and ground-truth regions,
normalised by their combined size, per class. Pixel accuracy is the fraction of pixels labelled
correctly *overall*, which on this dataset is close to useless: predicting background everywhere
scores 72% accuracy and DSC 0.0 on all three real classes. DSC is computed per class and cannot
be inflated by the majority class.

**Why one-hot / categorical output?**
The final 1×1 conv emits 4 channels — one logit per class per pixel. Softmax over the channel
axis gives a proper per-pixel probability distribution, and `argmax` gives the label. The
alternative, regressing a single channel to values 0–3, would falsely imply the classes are
ordered and that class 1 is "between" 0 and 2, which is anatomically meaningless. A 1×1 kernel is
right because all spatial reasoning has already happened; this step just projects each pixel's
feature vector onto the class axes.

**Why is the smallest class the hardest?**
Three compounding reasons. It supplies the fewest training examples per epoch. It contributes
least to any pixel-averaged loss, so it is the first thing a model neglects. And it has the worst
perimeter-to-area ratio, so a fixed one-pixel boundary error costs proportionally far more of its
DSC than the same error costs a large class — which is also why resolution matters most for it.

**Why NEAREST interpolation for the masks?**
Bilinear averages neighbouring pixels. On an image that is correct; on a label map it invents
values — averaging class 1 (85) and class 2 (170) gives 127, which decodes to a class that never
existed. NEAREST copies a single original pixel, so only real labels survive.

**Why does the checkpoint select on the worst class rather than the mean?**
The requirement is > 0.9 on *every* label. A model with mean DSC 0.94 but class 1 at 0.86 fails,
while one with mean 0.92 and minimum 0.91 passes. Selecting on the minimum optimises the thing
that actually determines the outcome.

**What would you try if class 1 falls short of 0.9?**
In order: raise `UNET_IMG` to 256 (boundary pixels are where the loss is, and class 1 suffers
most); increase `UNET_DICE_W` relative to `UNET_CE_W` to push harder on class balance; add class
weights to the CE term; widen the network with `UNET_BASE=64`; or train longer.